In [ ]:
# 1. Installa le librerie necessarie (eseguito una volta per sessione)
!pip install torchlibrosa pretty_midi mir_eval huggingface_hub -q

# 2. Import
import os
import random
import logging
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
import pretty_midi
from torch.utils.data import Dataset, DataLoader

# 3. Funzione mancante che avevi dimenticato di incollare
def parse_midi_to_events(midi_path):
    """Legge un file MIDI e restituisce una lista di eventi nota ordinati."""
    midi_data = pretty_midi.PrettyMIDI(str(midi_path))
    events = []
    for instrument in midi_data.instruments:
        if instrument.is_drum: continue
        for note in instrument.notes:
            events.append({
                "start": float(note.start),
                "end": float(note.end),
                "pitch": int(note.pitch),
                "velocity": int(note.velocity)
            })
    events.sort(key=lambda x: x["start"])
    return events

# 4. Il resto del tuo codice (che è perfetto)
def get_regression(center_frame_float, num_frames, left_frames=3, right_frames=3):
    """Rampa che codifica lo scarto sub-frame (stile Kong)."""
    target = np.zeros(num_frames, dtype=np.float32)
    step = 1.0 / (right_frames + 1)

    center_int = int(round(center_frame_float))
    shift = center_frame_float - center_int     # ora in [-0.5, +0.5], coerente col centro

    for i in range(-left_frames, right_frames + 1):
        idx = center_int + i
        if 0 <= idx < num_frames:
            dist = abs(i - shift)               # distanza dal vero onset, non dal frame arrotondato
            target[idx] = max(0.0, 1.0 - dist * step)
    return target

def create_targets_from_midi_events(midi_events, num_frames, classes_num, begin_note, frames_per_second):
    frame_target = np.zeros((num_frames, classes_num), dtype=np.float32)
    onset_target = np.zeros((num_frames, classes_num), dtype=np.float32)
    offset_target = np.zeros((num_frames, classes_num), dtype=np.float32)
    velocity_target = np.zeros((num_frames, classes_num), dtype=np.float32)
    reg_onset_target = np.zeros((num_frames, classes_num), dtype=np.float32)
    reg_offset_target = np.zeros((num_frames, classes_num), dtype=np.float32)

    for event in midi_events:
        pitch = event["pitch"] - begin_note
        if pitch < 0 or pitch >= classes_num: continue

        # Usa i float per calcolare la posizione esatta sub-frame
        start_frame_float = event["start"] * frames_per_second
        end_frame_float = event["end"] * frames_per_second
        start_frame = int(round(start_frame_float))
        end_frame = int(round(end_frame_float))
        start_frame = max(0, min(num_frames - 1, start_frame))
        end_frame = max(0, min(num_frames - 1, end_frame))
        if start_frame == end_frame: end_frame = min(num_frames - 1, start_frame + 1)

        frame_target[start_frame : end_frame + 1, pitch] = 1.0
        onset_target[start_frame, pitch] = 1.0
        offset_target[end_frame, pitch] = 1.0

        # Velocity: /128 e messa solo sul frame di onset (come Kong)
        velocity_target[start_frame, pitch] = event["velocity"] / 128.0

        # Regression sub-frame (asimmetriche, dipendenti dallo shift)
        reg_onset_target[:, pitch] = get_regression(start_frame_float, num_frames)
        reg_offset_target[:, pitch] = get_regression(end_frame_float, num_frames)

    return {
        "frame_target": frame_target, "onset_target": onset_target, "offset_target": offset_target,
        "velocity_target": velocity_target, "reg_onset_target": reg_onset_target, "reg_offset_target": reg_offset_target
    }

class GapsDataset(Dataset):
    def __init__(self, pairs, segment_seconds, hop_seconds, augment=False):
        self.pairs = pairs
        self.segment_seconds = segment_seconds
        self.hop_seconds = hop_seconds
        self.augment = augment
        self.segment_samples = int(segment_seconds * config.SAMPLE_RATE)
        # FIX: num_output_frames a 100 fps (1000 frame per 10 secondi)
        self.num_output_frames = round(segment_seconds * config.OUTPUT_FPS) + 1
        self.segments = self._build_segment_index()

    def _build_segment_index(self):
        segments = []
        for pair_idx, (audio_path, _) in enumerate(self.pairs):
            try: duration = librosa.get_duration(path=str(audio_path))
            except: continue
            if duration < self.segment_seconds: segments.append((pair_idx, 0.0))
            else:
                start = 0.0
                while start + self.segment_seconds <= duration:
                    segments.append((pair_idx, start))
                    start += self.hop_seconds
        return segments

    def __len__(self): return len(self.segments)

    def __getitem__(self, idx):
        pair_idx, start_time = self.segments[idx]
        audio_path, midi_path = self.pairs[pair_idx]
        waveform, _ = librosa.load(str(audio_path), sr=config.SAMPLE_RATE, mono=True, offset=start_time, duration=self.segment_seconds)
        if len(waveform) < self.segment_samples: waveform = np.pad(waveform, (0, self.segment_samples - len(waveform)))
        elif len(waveform) > self.segment_samples: waveform = waveform[:self.segment_samples]

        events = parse_midi_to_events(midi_path)
        segment_events = [{"start": max(0, e["start"] - start_time), "end": min(self.segment_seconds, e["end"] - start_time),
                           "pitch": e["pitch"], "velocity": e["velocity"]} for e in events if e["end"] > start_time and e["start"] < start_time + self.segment_seconds]

        targets = create_targets_from_midi_events(segment_events, self.num_output_frames, config.CLASSES_NUM, config.BEGIN_NOTE, config.OUTPUT_FPS)

        if self.augment:
            shift = random.randint(-2, 2)
            if shift != 0:
                waveform = librosa.effects.pitch_shift(waveform, sr=config.SAMPLE_RATE, n_steps=shift)
                for k in targets.keys(): targets[k] = np.roll(targets[k], shift, axis=1)
                if shift > 0:
                    for k in targets.keys(): targets[k][:, :shift] = 0
                elif shift < 0:
                    for k in targets.keys(): targets[k][:, shift:] = 0
            if random.random() < 0.5:
                gain_db = random.uniform(-6.0, 6.0); waveform = waveform * (10.0 ** (gain_db / 20.0))
            waveform = np.clip(waveform, -1.0, 1.0)

        sample = {"waveform": torch.from_numpy(waveform).float()}
        for k in targets: sample[k] = torch.from_numpy(targets[k]).float()
        return sample

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 111.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 6.1 MB/s eta 0:00:00


In [ ]:
import os
import random
import logging
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
import pretty_midi
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import LambdaLR
from torch.cuda.amp import GradScaler, autocast

# Configurazione logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device attivo: {device}")

class Config:
    # Percorsi Colab
    WORK_DIR = Path("/content/")
    DATA_DIR = WORK_DIR / "gaps_data"
    GS_DATA_DIR = WORK_DIR / "guitarset_data"
    MAESTRO_CHECKPOINT = WORK_DIR / "CRNN_note_F1=0.9677_pedal_F1=0.9186.pth"
    BEST_MODEL_PATH = WORK_DIR / "best_guitar_model.pth"

    # Audio
    SAMPLE_RATE = 16000
    WINDOW_SIZE = 2048
    HOP_SIZE = 160
    MEL_BINS = 229
    FMIN = 30
    FMAX = SAMPLE_RATE // 2
    WINDOW_TYPE = "hann"
    CENTER = True
    PAD_MODE = "reflect"

    # Modello
    CLASSES_NUM = 88
    BEGIN_NOTE = 21
    FRAMES_PER_SECOND = SAMPLE_RATE // HOP_SIZE
    OUTPUT_FPS = FRAMES_PER_SECOND
    MOMENTUM = 0.01
    MIDFEAT = 1792

    # Training (Ottimizzato per Colab Pro+ - A100/L4)
    SEGMENT_SECONDS = 10.0
    BATCH_SIZE = 4      # Abbassato da 32 a 16 per evitare OOM con la regressione attiva
    NUM_EPOCHS = 3      # ~30k-35k step totali (con GS+GAPS), come da paper
    LEARNING_RATE = 1e-5 # Abbassato come da paper Riley
    WEIGHT_DECAY = 1e-4
    PATIENCE = 8         # Early stopping

    # Freeze (Scaldiamo il BatchNorm per 1 epoca)
    FREEZE_EPOCHS = 0
    FREEZE_CONV_BLOCKS = 0

config = Config()

# Abilita Mixed Precision (AMP) per velocizzare il training del 2x su A100
scaler = GradScaler()

Device attivo: cuda


/tmp/ipykernel_1906/16960350.py:67: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
"""
model.py — Architettura CRNN per Automatic Music Transcription.

Implementazione fedele del modello 'Regress_onset_offset_frame_velocity_CRNN'
dal paper di Kong et al. (ByteDance):
  "High-resolution Piano Transcription with Pedals by Regressing Onset and Offset Times"

Questa architettura è compatibile con i pesi pre-addestrati sul dataset MAESTRO.
I nomi dei layer DEVONO corrispondere esattamente al checkpoint per il caricamento.

Componenti principali:
  1. Front-end: Spectrogram → LogMelFilterBank (via torchlibrosa)
  2. 4× AcousticModelCRnn indipendenti (frame, onset, offset, velocity)
  3. 2× GRU+FC di regressione per onset/offset ad alta risoluzione

Ogni AcousticModelCRnn contiene:
  - 4× ConvBlock (1→48→64→96→128 canali)
  - FC (3584→768) + BatchNorm1d
  - Bi-GRU (768→256×2=512)
  - FC (512→classes_num)
"""

import math
import logging
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchlibrosa.stft import Spectrogram, LogmelFilterBank

logger = logging.getLogger(__name__)


# =============================================================================
# Funzioni di inizializzazione pesi (identiche a Kong et al.)
# =============================================================================

def init_layer(layer):
    """Inizializza un layer Linear o Convoluzionale con Xavier uniform."""
    nn.init.xavier_uniform_(layer.weight)
    if hasattr(layer, "bias"):
        if layer.bias is not None:
            layer.bias.data.fill_(0.0)


def init_bn(bn):
    """Inizializza un layer BatchNorm."""
    bn.bias.data.fill_(0.0)
    bn.weight.data.fill_(1.0)


def init_gru(rnn):
    """Inizializza un layer GRU con schema specifico per AMT."""

    def _concat_init(tensor, init_funcs):
        (length, fan_out) = tensor.shape
        fan_in = length // len(init_funcs)
        for (i, init_func) in enumerate(init_funcs):
            init_func(tensor[i * fan_in : (i + 1) * fan_in, :])

    def _inner_uniform(tensor):
        fan_in = nn.init._calculate_correct_fan(tensor, "fan_in")
        nn.init.uniform_(tensor, -math.sqrt(3 / fan_in), math.sqrt(3 / fan_in))

    for i in range(rnn.num_layers):
        _concat_init(
            getattr(rnn, "weight_ih_l{}".format(i)),
            [_inner_uniform, _inner_uniform, _inner_uniform],
        )
        torch.nn.init.constant_(getattr(rnn, "bias_ih_l{}".format(i)), 0)

        _concat_init(
            getattr(rnn, "weight_hh_l{}".format(i)),
            [_inner_uniform, _inner_uniform, nn.init.orthogonal_],
        )
        torch.nn.init.constant_(getattr(rnn, "bias_hh_l{}".format(i)), 0)

        # Se bidirezionale, inizializza anche i pesi reverse
        if rnn.bidirectional:
            _concat_init(
                getattr(rnn, "weight_ih_l{}_reverse".format(i)),
                [_inner_uniform, _inner_uniform, _inner_uniform],
            )
            torch.nn.init.constant_(
                getattr(rnn, "bias_ih_l{}_reverse".format(i)), 0
            )
            _concat_init(
                getattr(rnn, "weight_hh_l{}_reverse".format(i)),
                [_inner_uniform, _inner_uniform, nn.init.orthogonal_],
            )
            torch.nn.init.constant_(
                getattr(rnn, "bias_hh_l{}_reverse".format(i)), 0
            )


# =============================================================================
# ConvBlock — Blocco convoluzionale base
# =============================================================================

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, momentum):
        super(ConvBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, (3, 3), (1, 1), (1, 1), bias=False)
        self.conv2 = nn.Conv2d(out_channels, out_channels, (3, 3), (1, 1), (1, 1), bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels, momentum); self.bn2 = nn.BatchNorm2d(out_channels, momentum)
        self.init_weight()
    def init_weight(self): init_layer(self.conv1); init_layer(self.conv2); init_bn(self.bn1); init_bn(self.bn2)
    def forward(self, input, pool_size=(2, 2), pool_type="avg"):
        x = F.relu_(self.bn1(self.conv1(input))); x = F.relu_(self.bn2(self.conv2(x)))
        if pool_type == "max": x = F.max_pool2d(x, kernel_size=pool_size)
        elif pool_type == "avg": x = F.avg_pool2d(x, kernel_size=pool_size)
        elif pool_type == "avg+max": x = F.avg_pool2d(x, kernel_size=pool_size) + F.max_pool2d(x, kernel_size=pool_size)
        return x

class AcousticModelCRnn(nn.Module):
    def __init__(self, classes_num, midfeat, momentum):
        super(AcousticModelCRnn, self).__init__()
        self.conv_block1 = ConvBlock(1, 48, momentum)
        self.conv_block2 = ConvBlock(48, 64, momentum)
        self.conv_block3 = ConvBlock(64, 96, momentum)
        self.conv_block4 = ConvBlock(96, 128, momentum)
        self.fc5 = nn.Linear(midfeat, 768, bias=False); self.bn5 = nn.BatchNorm1d(768, momentum)
        self.gru = nn.GRU(768, 256, num_layers=2, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(512, classes_num, bias=True)
        self.init_weight()
    def init_weight(self): init_layer(self.fc5); init_bn(self.bn5); init_gru(self.gru); init_layer(self.fc)
    def forward(self, input):
        # FIX KILLER #1: pool_size=(1, 2) su tutti i blocchi per mantenere 100 fps
        x = self.conv_block1(input, pool_size=(1, 2), pool_type="avg")
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block2(x, pool_size=(1, 2), pool_type="avg")
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block3(x, pool_size=(1, 2), pool_type="avg")
        x = F.dropout(x, p=0.2, training=self.training)
        x = self.conv_block4(x, pool_size=(1, 2), pool_type="avg")
        x = F.dropout(x, p=0.2, training=self.training)

        x = x.transpose(1, 2).flatten(2)
        x = self.fc5(x); x = x.transpose(1, 2); x = self.bn5(x); x = x.transpose(1, 2)
        x = F.relu(x); x = F.dropout(x, p=0.5, training=self.training)
        x, _ = self.gru(x); x = self.fc(x)
        return x

class Regress_onset_offset_frame_velocity_CRNN(nn.Module):
    def __init__(self, frames_per_second=None, classes_num=None):
        super(Regress_onset_offset_frame_velocity_CRNN, self).__init__()
        if frames_per_second is None: frames_per_second = config.FRAMES_PER_SECOND
        if classes_num is None: classes_num = config.CLASSES_NUM
        sample_rate, window_size, hop_size = config.SAMPLE_RATE, config.WINDOW_SIZE, config.HOP_SIZE
        mel_bins, fmin, fmax = config.MEL_BINS, config.FMIN, config.FMAX
        momentum, midfeat = config.MOMENTUM, config.MIDFEAT

        self.spectrogram_extractor = Spectrogram(n_fft=window_size, hop_length=hop_size, win_length=window_size, window=config.WINDOW_TYPE, center=config.CENTER, pad_mode=config.PAD_MODE, freeze_parameters=True)
        self.logmel_extractor = LogmelFilterBank(sr=sample_rate, n_fft=window_size, n_mels=mel_bins, fmin=fmin, fmax=fmax, ref=1.0, amin=1e-10, top_db=None, freeze_parameters=True)
        self.bn0 = nn.BatchNorm2d(mel_bins, momentum)

        self.frame_model = AcousticModelCRnn(classes_num, midfeat, momentum)
        self.reg_onset_model = AcousticModelCRnn(classes_num, midfeat, momentum)
        self.reg_offset_model = AcousticModelCRnn(classes_num, midfeat, momentum)
        self.velocity_model = AcousticModelCRnn(classes_num, midfeat, momentum)

        # FIX KILLER #3: Nomi esatti di Kong e condizionamento corretto
        self.reg_onset_gru = nn.GRU(input_size=classes_num * 2, hidden_size=256, num_layers=1, batch_first=True, bidirectional=True)
        self.reg_onset_fc = nn.Linear(512, classes_num, bias=True)
        self.frame_gru = nn.GRU(input_size=classes_num * 3, hidden_size=256, num_layers=1, batch_first=True, bidirectional=True) # Si chiama frame_gru in Kong
        self.frame_fc = nn.Linear(512, classes_num, bias=True) # Si chiama frame_fc in Kong
        self.init_weight()

    def init_weight(self): init_bn(self.bn0); init_gru(self.reg_onset_gru); init_layer(self.reg_onset_fc); init_gru(self.frame_gru); init_layer(self.frame_fc)
    def forward(self, input):
        x = self.spectrogram_extractor(input); x = self.logmel_extractor(x)
        x = x.transpose(1, 3); x = self.bn0(x); x = x.transpose(1, 3)

        # 1. Output grezzi dai 4 modelli acustici
        frame_output = torch.sigmoid(self.frame_model(x))
        reg_onset_output = torch.sigmoid(self.reg_onset_model(x))
        reg_offset_output = torch.sigmoid(self.reg_offset_model(x))
        velocity_output = torch.sigmoid(self.velocity_model(x))

        # 2. Raffinamento Onset (condizionato su sqrt(onset) * velocity)
        x = torch.cat((reg_onset_output, (reg_onset_output ** 0.5) * velocity_output.detach()), dim=2)
        (x, _) = self.reg_onset_gru(x)
        x = F.dropout(x, p=0.5, training=self.training)
        reg_onset_output = torch.sigmoid(self.reg_onset_fc(x))

        # 3. Raffinamento Frame (condizionato su onset e offset raffinati/grezzi)
        x = torch.cat((frame_output, reg_onset_output.detach(), reg_offset_output.detach()), dim=2)
        (x, _) = self.frame_gru(x)
        x = F.dropout(x, p=0.5, training=self.training)
        frame_output = torch.sigmoid(self.frame_fc(x))

        # 4. reg_offset_output NON viene toccato: resta quello dell'acoustic model

        # 5. Output dict
        output_dict = {
            "reg_onset_output": reg_onset_output,
            "reg_offset_output": reg_offset_output,
            "frame_output": frame_output,
            "velocity_output": velocity_output,
            "onset_output": reg_onset_output,
            "offset_output": reg_offset_output,
        }
        return output_dict

# FIX KILLER #3: Load checkpoint senza rename e con strict=True
def load_maestro_checkpoint(model, checkpoint_path, device="cpu"):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state_dict = checkpoint.get("model", checkpoint)
    if "note_model" in state_dict: state_dict = state_dict["note_model"]

    # Togliamo il rename e usiamo strict=True.
    # Se esplode qui, significa che i nomi dei layer nel model.py non combaciano al 100% col checkpoint.
    model.load_state_dict(state_dict, strict=True)

    model.to(device)
    return model


def load_finetuned_checkpoint(
    model: Regress_onset_offset_frame_velocity_CRNN,
    checkpoint_path: str | Path,
    device: str = "cpu",
) -> Regress_onset_offset_frame_velocity_CRNN:
    """
    Carica un checkpoint fine-tuned (salvato dal nostro training script).

    Args:
        model: Istanza del modello CRNN.
        checkpoint_path: Percorso al file .pth del checkpoint fine-tuned.
        device: Device su cui caricare i pesi.

    Returns:
        Il modello con i pesi caricati.
    """
    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.exists():
        raise FileNotFoundError(f"Checkpoint non trovato: {checkpoint_path}")

    logger.info(f"Caricamento checkpoint fine-tuned da: {checkpoint_path}")

    checkpoint = torch.load(
        checkpoint_path, map_location=device, weights_only=False
    )

    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        epoch = checkpoint.get("epoch", "?")
        val_loss = checkpoint.get("val_loss", "?")
        logger.info(f"Checkpoint epoca {epoch}, val_loss={val_loss}")
    else:
        model.load_state_dict(checkpoint)

    model.to(device)
    model.eval()

    return model


def verify_checkpoint_compatibility(
    model: Regress_onset_offset_frame_velocity_CRNN,
    checkpoint_path: str | Path,
    device: str = "cpu",
) -> dict:
    """
    Verifica la compatibilità tra il modello e un checkpoint,
    senza caricare effettivamente i pesi.

    Utile per diagnosticare problemi di incompatibilità prima del training.

    Returns:
        dict con:
            'model_keys': set delle chiavi del modello
            'checkpoint_keys': set delle chiavi del checkpoint
            'missing': chiavi nel modello ma non nel checkpoint
            'unexpected': chiavi nel checkpoint ma non nel modello
            'compatible': True se tutte le chiavi del modello sono nel checkpoint
    """
    checkpoint_path = Path(checkpoint_path)
    checkpoint = torch.load(
        checkpoint_path, map_location=device, weights_only=False
    )

    if isinstance(checkpoint, dict):
        if "model" in checkpoint:
            state_dict = checkpoint["model"]
        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        elif "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint

    model_keys = set(model.state_dict().keys())
    checkpoint_keys = set(state_dict.keys())

    missing = model_keys - checkpoint_keys
    unexpected = checkpoint_keys - model_keys

    result = {
        "model_keys": model_keys,
        "checkpoint_keys": checkpoint_keys,
        "missing": missing,
        "unexpected": unexpected,
        "compatible": len(missing) == 0,
    }

    if result["compatible"]:
        logger.info("✓ Checkpoint compatibile al 100% con il modello.")
    else:
        logger.warning(
            f"✗ Checkpoint parzialmente compatibile.\n"
            f"  Chiavi mancanti: {len(missing)}\n"
            f"  Chiavi inattese: {len(unexpected)}"
        )

    return result


In [ ]:
import os
import random
import urllib.request
from pathlib import Path
from huggingface_hub import snapshot_download
from torch.utils.data import DataLoader

# 1. Scarica GAPS (se non presente)
if not config.DATA_DIR.exists():
    print("Scaricamento GAPS dataset...")
    config.DATA_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(repo_id="xavriley/GAPS", repo_type="dataset",
                      local_dir=str(config.DATA_DIR), allow_patterns=["audio/*.wav", "midi/*.mid"])

# 2. Trova coppie GAPS
audio_dir = config.DATA_DIR / "audio"
midi_dir = config.DATA_DIR / "midi"
gaps_pairs = [(audio_dir / f.name.replace(".mid", ".wav"), f) for f in midi_dir.glob("*.mid")]
gaps_pairs = [(a, m) for a, m in gaps_pairs if a.exists()]
print(f"Trovate {len(gaps_pairs)} coppie audio/midi GAPS.")

# 3. Split 80/10/10
random.shuffle(gaps_pairs)
n_total = len(gaps_pairs)
n_train = int(n_total * 0.8)
n_val = int(n_total * 0.1)
train_pairs = gaps_pairs[:n_train]
val_pairs = gaps_pairs[n_train:n_train+n_val]
test_pairs = gaps_pairs[n_train+n_val:]

# 4. Creazione Datasets (Hop 1 secondo peqr massimizzare i dati nel training)
train_ds = GapsDataset(train_pairs, config.SEGMENT_SECONDS, hop_seconds=1.0, augment=True)
val_ds = GapsDataset(val_pairs, config.SEGMENT_SECONDS, hop_seconds=config.SEGMENT_SECONDS, augment=False)

# 5. DataLoader
train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train segments: {len(train_ds)} (≈{len(train_ds)//config.BATCH_SIZE} steps/epoch)")
print(f"Val segments: {len(val_ds)}")

Scaricamento GAPS dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]

Trovate 401 coppie audio/midi GAPS.
Train segments: 64317 (≈16079 steps/epoch)
Val segments: 843


In [ ]:
import urllib.request
import os

# Percorso dove il modello si aspetta di trovare il file (nella cartella /content/ di Colab)
target_path = "/content/CRNN_note_F1=0.9677_pedal_F1=0.9186.pth"
url = "https://zenodo.org/record/4034264/files/CRNN_note_F1%3D0.9677_pedal_F1%3D0.9186.pth?download=1"

if not os.path.exists(target_path) or os.path.getsize(target_path) < 1_000_000:
    print("Download dei pesi MAESTRO da Zenodo in corso...")
    urllib.request.urlretrieve(url, target_path)
    print(f"✓ Download completato! Dimensione: {os.path.getsize(target_path) / 1e6:.2f} MB")
else:
    print("✓ I pesi MAESTRO sono già presenti e validi.")

Download dei pesi MAESTRO da Zenodo in corso...
✓ Download completato! Dimensione: 171.97 MB


In [ ]:
import os
import time
import torch
import torch.nn as nn
from torch.optim.lr_scheduler import LambdaLR
from torch.amp import GradScaler, autocast   # API nuova, niente più deprecation warning

# ==========================================
# 1. MOUNT DI GOOGLE DRIVE
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

CHECKPOINT_DIR = "/content/drive/MyDrive/Colab_Checkpoints/Guitar_AMT"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "training_checkpoint.pth")
BEST_MODEL_PATH = os.path.join(CHECKPOINT_DIR, "best_guitar_model.pth")

# ==========================================
# 2. LOSS FUNCTION (calcolata in fp32, a prova di NaN sotto AMP)
# ==========================================
class TranscriptionLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, output_dict, target_dict):
        # Isola fp32: anche se il forward gira in AMP (fp16), la BCE qui è in fp32.
        with autocast(device_type='cuda', enabled=False):
            frame_output      = output_dict["frame_output"].float()
            reg_onset_output  = output_dict["reg_onset_output"].float()
            reg_offset_output = output_dict["reg_offset_output"].float()
            velocity_output   = output_dict["velocity_output"].float()

            frame_target      = target_dict["frame_target"].float()
            reg_onset_target  = target_dict["reg_onset_target"].float()
            reg_offset_target = target_dict["reg_offset_target"].float()
            velocity_target   = target_dict["velocity_target"].float()
            onset_target      = target_dict["onset_target"].float()

            def bce(output, target):
                output = torch.clamp(output, 1e-7, 1.0 - 1e-7)
                return -target * torch.log(output) - (1.0 - target) * torch.log(1.0 - output)

            frame_loss      = torch.mean(bce(frame_output, frame_target))
            reg_onset_loss  = torch.mean(bce(reg_onset_output, reg_onset_target))
            reg_offset_loss = torch.mean(bce(reg_offset_output, reg_offset_target))

            # Velocity: BCE mascherata sugli onset
            v_mask = onset_target
            v_loss_raw = bce(velocity_output, velocity_target)
            velocity_loss = (v_loss_raw * v_mask).sum() / v_mask.sum().clamp(min=1.0)

            return reg_onset_loss + reg_offset_loss + frame_loss + velocity_loss

# ==========================================
# 3. OPTIMIZER & SCHEDULER
# ==========================================
def build_optimizer_and_scheduler(model, total_steps):
    """AdamW + step decay 0.9 spalmato su 10 tappe lungo l'intero training."""
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)

    step_size = max(1, total_steps // 10)   # 10 decadimenti dall'inizio alla fine
    def lr_lambda(current_step):
        return 0.9 ** (current_step / step_size)

    scheduler = LambdaLR(optimizer, lr_lambda)
    return optimizer, scheduler

# ==========================================
# 4. CHECKPOINTING
# ==========================================
def save_checkpoint(epoch, model, optimizer, scheduler, scaler, best_val_loss, patience_counter):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'best_val_loss': best_val_loss,
        'patience_counter': patience_counter
    }, CHECKPOINT_PATH)

def load_checkpoint(model, optimizer, scheduler, scaler):
    if os.path.exists(CHECKPOINT_PATH):
        print("Trovato checkpoint esistente su Drive. Riprendendo il training...")
        checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_loss = checkpoint['best_val_loss']
        patience_counter = checkpoint['patience_counter']
        print(f"  -> Ripreso dall'epoch {start_epoch}. Best Val Loss: {best_val_loss:.4f}")
        return start_epoch, best_val_loss, patience_counter
    else:
        print("Nessun checkpoint trovato. Partenza da zero.")
        return 1, float('inf'), 0

# ==========================================
# 5. TRAINING DI UN'EPOCA (AMP nel forward, loss in fp32)
# ==========================================
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler, device, scaler=None):
    model.train()
    total_loss = 0
    for batch_idx, batch in enumerate(loader):
        waveform = batch["waveform"].to(device, non_blocking=True)
        target_dict = {k: v.to(device, non_blocking=True) for k, v in batch.items() if k != "waveform"}

        optimizer.zero_grad(set_to_none=True)

        output_dict = model(waveform)          # tutto fp32, niente autocast
        loss = loss_fn(output_dict, target_dict)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()                        # dopo optimizer.step(), vedi nota sotto

        total_loss += loss.item()
        if batch_idx % 50 == 0:
            print(f"  Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f} | LR: {scheduler.get_last_lr()[0]:.2e}")
    return total_loss / len(loader)

# ==========================================
# 6. SETUP MODELLO
# ==========================================
print("Inizializzazione modello...")
model = Regress_onset_offset_frame_velocity_CRNN(
    frames_per_second=config.FRAMES_PER_SECOND,
    classes_num=config.CLASSES_NUM
).to(device)

assert os.path.exists(config.MAESTRO_CHECKPOINT), "Checkpoint MAESTRO mancante! Scaricalo prima."
model = load_maestro_checkpoint(model, config.MAESTRO_CHECKPOINT, device=device)
print("✅ Checkpoint MAESTRO caricato.")

for param in model.parameters():
    param.requires_grad = True

total_steps = len(train_loader) * config.NUM_EPOCHS
optimizer, scheduler = build_optimizer_and_scheduler(model, total_steps)
scaler = GradScaler(device='cuda')
loss_fn = TranscriptionLoss()

start_epoch, best_val_loss, patience_counter = load_checkpoint(model, optimizer, scheduler, scaler)

# ==========================================
# 7. TRAINING LOOP
# ==========================================
print(f"\nInizio Training da Epoch {start_epoch} fino a {config.NUM_EPOCHS}...")
print(f"Step totali previsti: {total_steps} (riferimento paper: ~30k)")

for epoch in range(start_epoch, config.NUM_EPOCHS + 1):
    print(f"\n--- Epoch {epoch}/{config.NUM_EPOCHS} ---")
    start_time = time.time()

    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, scheduler, device, scaler)

    # Validation — STESSA precisione del training (AMP nel forward, loss fp32)
    model.eval()
    val_loss = 0
    with torch.no_grad():
      for batch in val_loader:
          waveform = batch["waveform"].to(device, non_blocking=True)
          target_dict = {k: v.to(device, non_blocking=True) for k, v in batch.items() if k != "waveform"}
          output_dict = model(waveform)
          val_loss += loss_fn(output_dict, target_dict).item()
    val_loss /= len(val_loader)

    elapsed_time = time.time() - start_time
    print(f"Epoch {epoch} riassunto | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {elapsed_time:.1f}s")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print("  -> ✅ Nuovo best model salvato!")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"  -> Val Loss non migliorata. Patience: {patience_counter}/{config.PATIENCE}")
        if patience_counter >= config.PATIENCE:
            print("  -> ⚠️ Early stopping attivato!")
            save_checkpoint(epoch, model, optimizer, scheduler, scaler, best_val_loss, patience_counter)
            break

    save_checkpoint(epoch, model, optimizer, scheduler, scaler, best_val_loss, patience_counter)

print("\n🎉 Training completato!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Inizializzazione modello...
✅ Checkpoint MAESTRO caricato.
Nessun checkpoint trovato. Partenza da zero.

Inizio Training da Epoch 1 fino a 3...
Step totali previsti: 48825 (riferimento paper: ~30k)

--- Epoch 1/3 ---
  Batch 0/16275 | Loss: 1.0450 | LR: 1.00e-05
  Batch 50/16275 | Loss: 0.8541 | LR: 9.99e-06
  Batch 100/16275 | Loss: 0.6606 | LR: 9.98e-06
  Batch 150/16275 | Loss: 0.6493 | LR: 9.97e-06
  Batch 200/16275 | Loss: 0.6509 | LR: 9.96e-06
  Batch 250/16275 | Loss: 0.5954 | LR: 9.95e-06
  Batch 300/16275 | Loss: 0.6427 | LR: 9.94e-06
  Batch 350/16275 | Loss: 0.6409 | LR: 9.92e-06
  Batch 400/16275 | Loss: 0.5882 | LR: 9.91e-06
  Batch 450/16275 | Loss: 0.6204 | LR: 9.90e-06
  Batch 500/16275 | Loss: 0.6359 | LR: 9.89e-06
  Batch 550/16275 | Loss: 0.6352 | LR: 9.88e-06
  Batch 600/16275 | Loss: 0.6166 | LR: 9.87e-06
  Batch 650/16275 | Loss: 0.6258 

In [ ]:
import numpy as np
import torch
import librosa
import pretty_midi
import mir_eval

@torch.no_grad()
def transcribe_full(model, wav_path, device, onset_thresh=0.3, frame_thresh=0.3,
                    segment_seconds=10.0, hop_seconds=10.0):
    """Trascrive un file intero a segmenti, decodificando con interpolazione sub-frame."""
    model.eval()
    fps = config.OUTPUT_FPS  # 100
    audio, _ = librosa.load(str(wav_path), sr=config.SAMPLE_RATE, mono=True)
    seg_samples = int(segment_seconds * config.SAMPLE_RATE)

    notes = []
    start = 0.0
    while start < len(audio) / config.SAMPLE_RATE:
        s0 = int(start * config.SAMPLE_RATE)
        chunk = audio[s0 : s0 + seg_samples]
        if len(chunk) < seg_samples:
            chunk = np.pad(chunk, (0, seg_samples - len(chunk)))
        wav_t = torch.from_numpy(chunk).float().unsqueeze(0).to(device)

        out = model(wav_t)
        onset = out["reg_onset_output"][0].cpu().numpy()    # (T, 88) rampe
        frame = out["frame_output"][0].cpu().numpy()
        vel   = out["velocity_output"][0].cpu().numpy()

        T, P = onset.shape
        for p in range(P):
            t = 1
            while t < T - 1:
                # picco locale sulla rampa di regressione
                if onset[t, p] >= onset_thresh and onset[t, p] >= onset[t-1, p] and onset[t, p] > onset[t+1, p]:
                    # interpolazione parabolica per il tempo sub-frame
                    a, b, c = onset[t-1, p], onset[t, p], onset[t+1, p]
                    denom = (a - 2*b + c)
                    shift = 0.5 * (a - c) / denom if abs(denom) > 1e-6 else 0.0
                    onset_time = start + (t + shift) / fps

                    # durata: estendi finché frame resta attivo
                    end_t = t + 1
                    while end_t < T and frame[end_t, p] > frame_thresh:
                        end_t += 1
                    end_time = start + end_t / fps
                    if end_time <= onset_time:
                        end_time = onset_time + 0.05

                    velocity = int(np.clip(vel[t, p] * 128, 1, 127))
                    notes.append((onset_time, end_time, p + config.BEGIN_NOTE, velocity))
                    t += 2  # refrattarietà
                else:
                    t += 1
        start += hop_seconds

    return notes

def evaluate_model(model, test_pairs, device):
    all_P, all_R, all_F = [], [], []
    print(f"{'file':<22} {'P':>6} {'R':>6} {'F1':>6} {'ref':>5} {'est':>5}")
    for wav_path, midi_path in test_pairs:
        est = transcribe_full(model, wav_path, device)
        ref_pm = pretty_midi.PrettyMIDI(str(midi_path))
        ref = [(n.start, n.end, n.pitch) for inst in ref_pm.instruments
               for n in inst.notes if not inst.is_drum]
        if len(ref) == 0 or len(est) == 0:
            continue

        ref_int = np.array([[s, e] for s, e, _ in ref])
        ref_pit = np.array([p for _, _, p in ref])
        est_int = np.array([[s, e] for s, e, _, _ in est])
        est_pit = np.array([p for _, _, p, _ in est])

        P, R, F1, _ = mir_eval.transcription.precision_recall_f1_overlap(
            ref_int, ref_pit, est_int, est_pit,
            onset_tolerance=0.05, pitch_tolerance=50.0, offset_ratio=None)
        all_P.append(P); all_R.append(R); all_F.append(F1)
        print(f"{wav_path.name:<22} {P*100:6.1f} {R*100:6.1f} {F1*100:6.1f} {len(ref):5d} {len(est):5d}")

    print("\n" + "="*48)
    print(f"  MEDIA TEST SET ({len(all_F)} file)")
    print(f"  Precision: {np.mean(all_P)*100:.2f}%  Recall: {np.mean(all_R)*100:.2f}%  F1: {np.mean(all_F)*100:.2f}%")
    print("="*48)

# ---- 1. SANITY CHECK: zero-shot (MAESTRO non allenato) ----
print(">>> ZERO-SHOT (checkpoint MAESTRO, atteso ~50 F1)")
zs_model = Regress_onset_offset_frame_velocity_CRNN(config.FRAMES_PER_SECOND, config.CLASSES_NUM).to(device)
zs_model = load_maestro_checkpoint(zs_model, config.MAESTRO_CHECKPOINT, device=device)
evaluate_model(zs_model, test_pairs, device)

# ---- 2. Il tuo modello fine-tuned ----
print("\n>>> FINE-TUNED (best_guitar_model.pth)")
BEST_MODEL_PATH = "/content/best_guitar_model.pth"   # definiscila qui, esplicita

ft_model = Regress_onset_offset_frame_velocity_CRNN(config.FRAMES_PER_SECOND, config.CLASSES_NUM).to(device)
ft_model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
evaluate_model(ft_model, test_pairs, device)

>>> ZERO-SHOT (checkpoint MAESTRO, atteso ~50 F1)
file                        P      R     F1   ref   est
388_Yk1wc.wav            34.7   31.7   33.1   189   173
052_hc1wc.wav            80.9   77.5   79.2   926   887
203_hk1wc.wav            53.8   96.3   69.1   431   771
337_6D1wc.wav            30.0    2.7    4.9   679    60
008_PSswc.wav            52.7   90.9   66.7   331   571
143_rD1wc.wav            45.4   88.1   59.9   386   749
033_LN1wc.wav            33.0   84.4   47.4   326   834
035_GN1wc.wav            55.3   88.0   68.0   300   477
057_MV1wc.wav            61.7   79.9   69.6   314   407
235_Ny1wc.wav            55.8   69.9   62.0  1589  1991
269_Hw1wc.wav            61.2   89.9   72.8   613   901
196_Lk1wc.wav            65.2   73.2   69.0   313   351
114_Sf1wc.wav            58.8   57.4   58.1   652   636
275_Lw1wc.wav            53.8   67.9   60.0  1375  1733
295_-Sswc.wav            75.4   53.6   62.7   578   411
307_5c1wc.wav            49.6   61.8   55.0   401   50

In [ ]:
for th in [0.05, 0.1, 0.15, 0.2, 0.3]:
    all_F, all_P, all_R = [], [], []
    for wav_path, midi_path in test_pairs:
        est = transcribe_full(ft_model, wav_path, device, onset_thresh=th, frame_thresh=0.3)
        ref_pm = pretty_midi.PrettyMIDI(str(midi_path))
        ref = [(n.start, n.end, n.pitch) for inst in ref_pm.instruments
               for n in inst.notes if not inst.is_drum]
        if len(ref) == 0 or len(est) == 0:
            continue
        ref_int = np.array([[s, e] for s, e, _ in ref]); ref_pit = np.array([p for _, _, p in ref])
        est_int = np.array([[s, e] for s, e, _, _ in est]); est_pit = np.array([p for _, _, p, _ in est])
        P, R, F1, _ = mir_eval.transcription.precision_recall_f1_overlap(
            ref_int, ref_pit, est_int, est_pit,
            onset_tolerance=0.05, pitch_tolerance=50.0, offset_ratio=None)
        all_P.append(P); all_R.append(R); all_F.append(F1)
    print(f"thresh={th:.2f} | P={np.mean(all_P)*100:5.1f}  R={np.mean(all_R)*100:5.1f}  F1={np.mean(all_F)*100:5.1f}")

thresh=0.05 | P= 69.8  R= 65.9  F1= 65.5
thresh=0.10 | P= 75.4  R= 52.0  F1= 58.9
thresh=0.15 | P= 78.1  R= 39.9  F1= 50.1


KeyboardInterrupt: 